# Maximum projected extent versus saved TiltDis over an eddy lifetime

Compare maximum projected extent with **the existing `surface.TiltDis` values**, loaded by `tilt.load_tilt_tables(paths, add_regions=True, grid=grid)`. No delta fit, smoothing or tilt recalculation is performed.

Only maximum projected extent is calculated: project the reference day's vertical centres onto the axis specified by its **saved `TiltDir`**, then take maximum minus minimum. This preserves the schematic's projected-extent definition, rather than maximum pairwise 2-D separation. The projection axis can change from day to day.

Choose `N_EDDIES` random eddies, change `SEED`, or provide `EDDY_IDS`. Saved tilt values remain plotted even on days without a usable vertical profile; the difference is calculated only where both measurements exist.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

HERE = Path.cwd().resolve()
REPO = next(p for p in (HERE, *HERE.parents) if (p / 'seacofs_eddy_dataset_modular').is_dir())
FOLDER = REPO / 'seacofs_eddy_tilt_analysis' / 'delta_tilt_method'
sys.path.insert(0, str(FOLDER.parent))
import seacofs_tilt_tools as tilt
paths = tilt.Paths()

PROFILE_PATH = paths.vert
SEED = 731
N_EDDIES = 30
EDDY_IDS = None             # e.g. [123, 456]; overrides N_EDDIES when supplied
MIN_VALID_DAYS = 50         # Minimum number of paired distance estimates per eddy
MAX_DEPTH = 1000
BEARING_OFFSET = 20.0      # Rotation used when saved TiltDir was generated
SAVE = False
OUT = FOLDER / 'extent_lifetime_outputs'
if SAVE:
    OUT.mkdir(exist_ok=True)

In [ ]:
if not isinstance(N_EDDIES, int) or isinstance(N_EDDIES, bool) or N_EDDIES < 1:
    raise ValueError('N_EDDIES must be a positive integer.')
if not isinstance(MIN_VALID_DAYS, int) or MIN_VALID_DAYS < 1:
    raise ValueError('MIN_VALID_DAYS must be a positive integer.')
grid = tilt.load_grid(paths.grid, paths.z_r)
surface, _ = tilt.load_tilt_tables(paths, add_regions=True, grid=grid)
profiles = pd.read_parquet(PROFILE_PATH, columns=['Eddy', 'Day', 'Depth', 'xc', 'yc'])
if surface[['Eddy', 'Day']].isna().any().any() or profiles[['Eddy', 'Day']].isna().any().any():
    raise ValueError('Missing Eddy/Day identifiers.')
if surface.duplicated(['Eddy', 'Day']).any():
    raise ValueError('Surface must have one row per Eddy-Day.')
counts = surface.groupby('Eddy').TiltDis.count()
if EDDY_IDS is None:
    # Uniform random order of eddy IDs; selection does not depend on distance mismatch.
    candidates = np.random.default_rng(SEED).permutation(counts[counts >= MIN_VALID_DAYS].index)
    requested = N_EDDIES
else:
    candidates = list(dict.fromkeys(EDDY_IDS))
    requested = len(candidates)
    if not candidates:
        raise ValueError('EDDY_IDS must contain at least one ID, or be None.')
    missing = set(candidates) - set(counts.index)
    if missing:
        raise ValueError(f'Eddy IDs absent from profiles: {sorted(missing)}')

series, selected = [], []
for eddy in candidates:
    eddy = int(eddy)
    track = surface.loc[surface.Eddy.eq(eddy), ['Eddy', 'Day', 'TiltDis', 'TiltDir']].copy()
    vertical = profiles.loc[profiles.Eddy.eq(eddy)].copy()
    vertical['Depth'] = vertical.Depth.abs()
    vertical = vertical.loc[vertical.Depth.le(MAX_DEPTH)]
    directions = track.set_index('Day').TiltDir
    rows = []
    for day, g in vertical.groupby('Day'):
        theta = directions.get(day, np.nan)
        xy = g[['xc', 'yc']].to_numpy()
        if len(g) < 2 or not np.isfinite(theta) or not np.isfinite(xy).all():
            continue
        # Saved TiltDir = native-grid bearing + BEARING_OFFSET.
        # Undo that offset to project native xc/yc (km). Axis reversal leaves the range unchanged.
        bearing = np.deg2rad(theta - BEARING_OFFSET)
        axis = np.array([np.sin(bearing), np.cos(bearing)])
        extent = float(np.ptp((xy - xy[0]) @ axis))
        rows.append({'Day': day, 'MaxProjectedExtent': extent, 'ReferenceDepthMax': g.Depth.max()})
    measured = pd.DataFrame(rows, columns=['Day', 'MaxProjectedExtent', 'ReferenceDepthMax'])
    paired = track.merge(measured, on='Day', how='left', validate='one_to_one')
    paired['Difference'] = paired.MaxProjectedExtent - paired.TiltDis
    paired['AbsoluteDifference'] = paired.Difference.abs()
    n_valid = int(np.isfinite(paired[['MaxProjectedExtent', 'TiltDis']]).all(axis=1).sum())
    if n_valid < MIN_VALID_DAYS:
        if EDDY_IDS is not None:
            print(f'Skipping eddy {eddy}: {n_valid} valid pairs; requires {MIN_VALID_DAYS}.')
        continue
    days = pd.DataFrame({'Day': np.arange(int(track.Day.min()), int(track.Day.max()) + 1)})
    daily = days.merge(paired, on='Day', how='left', validate='one_to_one')
    daily['Eddy'] = eddy
    daily['AgeDays'] = daily.Day - int(track.Day.min())
    series.append(daily)
    selected.append(eddy)
    print(f'Selected eddy {eddy}: {n_valid} paired days over {len(daily)} calendar days')
    if len(selected) == requested:
        break
if not series:
    raise ValueError('No eligible eddies. Reduce MIN_VALID_DAYS or choose different EDDY_IDS.')
if len(selected) < requested:
    print(f'Found {len(selected)} eligible eddies; requested {requested}.')
comparison = pd.concat(series, ignore_index=True)
print(f'To reproduce this selection: EDDY_IDS = {selected}')

## Lifetime comparison

The upper panel shows saved `surface.TiltDis` and the calculated maximum projected extent in kilometres. The lower panel shows **projected extent − saved TiltDis**. No additional smoothing is applied.

Age starts at the first available surface observation. Surface values are preserved as loaded; missing calendar days remain gaps. Projected extent requires a finite saved `TiltDir` and at least two valid vertical centres within `MAX_DEPTH`. No new temporal-window or delta-fit filters are applied. Changing `MAX_DEPTH` changes only projected extent, never saved `TiltDis`. Use the depth cap and `BEARING_OFFSET` appropriate to the saved tilt dataset.

In [ ]:
for eddy, daily in comparison.groupby('Eddy', sort=False):
    valid = daily.dropna(subset=['MaxProjectedExtent', 'TiltDis'])
    fig, axs = plt.subplots(2, 1, figsize=(11, 5), sharex=True,
        constrained_layout=True, gridspec_kw={'height_ratios': [2, 1]})
    axs[0].plot(daily.AgeDays, daily.MaxProjectedExtent, color='crimson',
                lw=1.8, label='Maximum projected extent')
    axs[0].plot(daily.AgeDays, daily.TiltDis, color='#222222',
                lw=1.8, label='Saved surface TiltDis')
    axs[0].set_title(f'Eddy {int(eddy)} · {len(valid)} paired days')
    axs[0].set_ylabel('Distance (km)')
    axs[0].legend()
    axs[1].plot(daily.AgeDays, daily.Difference, color='#3569a8', lw=1.5)
    axs[1].axhline(0, color='grey', lw=0.8)
    axs[1].set_ylabel('Extent − TiltDis (km)')
    axs[1].set_xlabel('Days since first surface observation')
    for ax in axs:
        ax.grid(alpha=0.2)
    if SAVE:
        fig.savefig(OUT / f'extent_vs_tilt_eddy_{int(eddy)}.png', dpi=180)
    plt.show()
    plt.close(fig)

summary = comparison.groupby('Eddy', sort=False).agg(
    PairedDays=('Difference', 'count'),
    MedianDifference_km=('Difference', 'median'),
    MedianAbsoluteDifference_km=('AbsoluteDifference', 'median'),
    MaxAbsoluteDifference_km=('AbsoluteDifference', 'max'))
display(summary.round(2))
if SAVE:
    comparison.to_csv(OUT / 'extent_vs_tilt_lifetimes.csv', index=False)
    summary.to_csv(OUT / 'extent_vs_tilt_summary.csv')

Maximum projected extent is a geometric comparison, not ground truth for the saved tilt estimate. Differences can reflect temporal smoothing in the saved estimate, curvature, fitting and depth coverage. `ReferenceDepthMax` records the deepest centre used to calculate each day's projected extent. The notebook does not modify the saved tilt dataset.